In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load data
def load_data(filepath='HR-Employee-Attrition.csv'):
    df = pd.read_csv(filepath)
    return df

# Data preprocessing
def preprocess_data(df):
    # Create a copy
    data = df.copy()
    
    # Drop unnecessary columns
    drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
    data = data.drop(columns=[col for col in drop_cols if col in data.columns])
    
    # Encode target variable
    data['Attrition'] = data['Attrition'].map({'Yes': 1, 'No': 0})
    
    # Identify categorical columns
    categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
    categorical_cols.remove('Attrition') if 'Attrition' in categorical_cols else None
    
    # Encode categorical variables
    le_dict = {}
    for col in categorical_cols:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])
        le_dict[col] = le
    
    # Separate features and target
    X = data.drop('Attrition', axis=1)
    y = data['Attrition']
    
    return X, y, le_dict, categorical_cols

# Feature engineering
def add_features(X):
    X = X.copy()
    # Add interaction features
    X['Age_YearsAtCompany'] = X['Age'] * X['YearsAtCompany']
    X['Income_Per_Year'] = X['MonthlyIncome'] / (X['YearsAtCompany'] + 1)
    X['JobSatisfaction_Environment'] = X['JobSatisfaction'] * X['EnvironmentSatisfaction']
    X['TotalWorkingYears_Ratio'] = X['TotalWorkingYears'] / (X['Age'] + 1)
    X['Promotion_Gap'] = X['YearsSinceLastPromotion'] / (X['YearsAtCompany'] + 1)
    X['WorkLife_Balance'] = X['WorkLifeBalance'] * X['JobInvolvement']
    return X

# Train model
def train_model(X, y):
    # Add engineered features
    X = add_features(X)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Get feature names
    feature_names = X_train.columns.tolist()
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Create a pipeline with Random Forest (best for this type of data)
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    print("=" * 60)
    print("MODEL EVALUATION")
    print("=" * 60)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'Feature': feature_names,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10))
    
    return model, scaler, feature_names

# Main execution
def main():
    print("Loading data...")
    df = load_data()
    print(f"Data shape: {df.shape}")
    
    print("\nPreprocessing data...")
    X, y, le_dict, categorical_cols = preprocess_data(df)
    print(f"Features: {X.shape[1]}")
    
    print("\nTraining model...")
    model, scaler, feature_names = train_model(X, y)
    
    # Save model and scaler
    print("\nSaving model...")
    joblib.dump(model, 'attrition_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')
    joblib.dump(feature_names, 'feature_names.pkl')
    joblib.dump(le_dict, 'label_encoders.pkl')
    print("Model saved successfully!")
    
    return model, scaler, feature_names, le_dict

if __name__ == "__main__":
    main()

Loading data...
Data shape: (1470, 35)

Preprocessing data...
Features: 30

Training model...
MODEL EVALUATION
Accuracy: 0.8197
ROC-AUC: 0.7732

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.93      0.90       247
           1       0.40      0.26      0.31        47

    accuracy                           0.82       294
   macro avg       0.63      0.59      0.60       294
weighted avg       0.79      0.82      0.80       294


Confusion Matrix:
[[229  18]
 [ 35  12]]

Top 10 Most Important Features:
                    Feature  Importance
30       Age_YearsAtCompany    0.076493
15            MonthlyIncome    0.064308
18                 OverTime    0.060968
0                       Age    0.052621
33  TotalWorkingYears_Ratio    0.051225
2                 DailyRate    0.043742
23        TotalWorkingYears    0.035930
26           YearsAtCompany    0.035809
31          Income_Per_Year    0.035701
35         WorkLife_Balance   